# ShopSphere Data Profiling

## Objective

The objective of this phase is to understand the structure,
completeness, consistency, and quality of the raw datasets.

We will not modify the raw data during profiling.

In [5]:
import pandas as pd
from pathlib import Path

## 1. Dataset Discovery

First, we identify all available CSV files in the raw data directory.

In [ ]:
RAW_DATA_PATH = Path("../Data/raw")

REPORT_PATH = Path("reports")



REPORT_PATH.mkdir(parents=True, exist_ok=True)

In [12]:
csv_files = list(RAW_DATA_PATH.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files.")

Found 11 CSV files.


## 2. Dataset Structure

In this section, we examine the basic structure of each dataset, including:

- Number of rows
- Number of columns
- Column names
- Data types
- Sample records

This helps us understand what information is available before performing
data quality checks or transformations.

In [14]:
dataset_profiles = []

for file in csv_files:
    df = pd.read_csv(file)

    dataset_profiles.append({
        "dataset": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

structure_df = pd.DataFrame(dataset_profiles)

structure_df

,dataset,rows,columns
0,olist_closed_deals_dataset.csv,842,14
1,olist_customers_dataset.csv,99441,5
2,olist_geolocation_dataset.csv,1000163,5
3,olist_marketing_qualified_leads_dataset.csv,8000,4
4,olist_orders_dataset.csv,99441,8
5,olist_order_items_dataset.csv,112650,7
6,olist_order_payments_dataset.csv,103886,5
7,olist_order_reviews_dataset.csv,99224,7
8,olist_products_dataset.csv,32951,9
9,olist_sellers_dataset.csv,3095,4


## Initial Observations

The datasets have different sizes and represent different business domains.

The `orders` dataset contains 99,441 records, while `order_items` contains
112,650 records. This indicates that an order may contain multiple order
items, but this relationship must be validated during referential integrity
analysis.

The `geolocation` dataset is significantly larger than the other datasets,
with more than one million records.

The datasets also have different levels of detail (grain). Understanding
the grain of each dataset is necessary before joining tables or calculating
business KPIs.

At this stage, these are observations only. No data-cleaning decisions are
being made.

## 3. Schema Profiling

This section examines the columns and data types available in each dataset.

The objective is to identify:

- Column names
- Data types
- Potential identifiers
- Numerical fields
- Categorical fields
- Date/time fields
- Fields that may act as primary or foreign keys

This information will be used later for data-quality validation,
transformation, and analysis.

In [15]:
schema_profiles = []

for file in csv_files:
    df = pd.read_csv(file)

    for column in df.columns:
        schema_profiles.append({
            "dataset": file.name,
            "column": column,
            "data_type": str(df[column].dtype),
            "non_null": df[column].notna().sum(),
            "null_count": df[column].isna().sum(),
            "unique_values": df[column].nunique()
        })

schema_df = pd.DataFrame(schema_profiles)

schema_df

,dataset,column,data_type,non_null,null_count,unique_values
0,olist_closed_deals_dataset.csv,mql_id,str,842,0,842
1,olist_closed_deals_dataset.csv,seller_id,str,842,0,842
2,olist_closed_deals_dataset.csv,sdr_id,str,842,0,32
3,olist_closed_deals_dataset.csv,sr_id,str,842,0,22
4,olist_closed_deals_dataset.csv,won_date,str,842,0,824
...,...,...,...,...,...,...
65,olist_sellers_dataset.csv,seller_zip_code_prefix,int64,3095,0,2246
66,olist_sellers_dataset.csv,seller_city,str,3095,0,611
67,olist_sellers_dataset.csv,seller_state,str,3095,0,23
68,product_category_name_translation.csv,product_category_name,str,71,0,71


## 4. Schema Report

The schema profile provides a consolidated view of all columns across the
available datasets.

The report includes the dataset name, column name, data type, non-null
count, null count, and number of unique values.

This report will be used as the foundation for subsequent data-quality
checks and transformation decisions.

In [16]:
schema_df.to_csv(
    REPORT_PATH / "schema_report.csv",
    index=False
)

print("Schema report saved successfully.")

Schema report saved successfully.


## 5. Potential Identifiers and Date Columns

Before performing data-quality validation, we identify columns that may
represent identifiers or dates.

These are potential candidates only and will be validated in later steps.

In [17]:
# Potential identifier columns
id_columns = schema_df[
    schema_df["column"].str.lower().str.contains("id")
]

id_columns

,dataset,column,data_type,non_null,null_count,unique_values
0,olist_closed_deals_dataset.csv,mql_id,str,842,0,842
1,olist_closed_deals_dataset.csv,seller_id,str,842,0,842
2,olist_closed_deals_dataset.csv,sdr_id,str,842,0,32
3,olist_closed_deals_dataset.csv,sr_id,str,842,0,22
14,olist_customers_dataset.csv,customer_id,str,99441,0,99441
15,olist_customers_dataset.csv,customer_unique_id,str,99441,0,96096
24,olist_marketing_qualified_leads_dataset.csv,mql_id,str,8000,0,8000
26,olist_marketing_qualified_leads_dataset.csv,landing_page_id,str,8000,0,495
28,olist_orders_dataset.csv,order_id,str,99441,0,99441
29,olist_orders_dataset.csv,customer_id,str,99441,0,99441


In [ ]:
# Potential date/time columns
date_columns = schema_df[
    schema_df["column"].str.lower().str.contains(
        "date|time"
    )
]

date_columns


,dataset,column,data_type,non_null,null_count,unique_values
4,olist_closed_deals_dataset.csv,won_date,str,842,0,824
25,olist_marketing_qualified_leads_dataset.csv,first_contact_date,str,8000,0,336
31,olist_orders_dataset.csv,order_purchase_timestamp,str,99441,0,98875
33,olist_orders_dataset.csv,order_delivered_carrier_date,str,97658,1783,81018
34,olist_orders_dataset.csv,order_delivered_customer_date,str,96476,2965,95664
35,olist_orders_dataset.csv,order_estimated_delivery_date,str,99441,0,459
40,olist_order_items_dataset.csv,shipping_limit_date,str,112650,0,93318
53,olist_order_reviews_dataset.csv,review_creation_date,str,99224,0,636
54,olist_order_reviews_dataset.csv,review_answer_timestamp,str,99224,0,98248


## 6. Primary Key Validation

Candidate primary keys identified during schema profiling are validated
for null values and duplicate records.

A column is considered a valid primary-key candidate only if:

- It contains no null values.
- Its values are unique within the dataset.

This validation is performed before establishing relationships between
datasets.

In [19]:
candidate_keys = {
    "olist_customers_dataset.csv": "customer_id",
    "olist_orders_dataset.csv": "order_id",
    "olist_products_dataset.csv": "product_id",
    "olist_sellers_dataset.csv": "seller_id",
    "olist_marketing_qualified_leads_dataset.csv": "mql_id",
    "olist_closed_deals_dataset.csv": "mql_id"
}

pk_results = []

for dataset, key in candidate_keys.items():

    file_path = RAW_DATA_PATH / dataset
    df = pd.read_csv(file_path)

    pk_results.append({
        "dataset": dataset,
        "candidate_key": key,
        "row_count": len(df),
        "null_count": df[key].isna().sum(),
        "unique_count": df[key].nunique(),
        "duplicate_count": df[key].duplicated().sum(),
        "is_valid_candidate": (
            df[key].notna().all()
            and df[key].is_unique
        )
    })

pk_validation_df = pd.DataFrame(pk_results)

pk_validation_df

,dataset,candidate_key,row_count,null_count,unique_count,duplicate_count,is_valid_candidate
0,olist_customers_dataset.csv,customer_id,99441,0,99441,0,True
1,olist_orders_dataset.csv,order_id,99441,0,99441,0,True
2,olist_products_dataset.csv,product_id,32951,0,32951,0,True
3,olist_sellers_dataset.csv,seller_id,3095,0,3095,0,True
4,olist_marketing_qualified_leads_dataset.csv,mql_id,8000,0,8000,0,True
5,olist_closed_deals_dataset.csv,mql_id,842,0,842,0,True


## Primary Key Validation Findings

All six identified candidate keys passed the initial primary-key validation.

Each candidate key contains:

- Zero null values
- Zero duplicate values
- A unique value for every record

Therefore, the following columns can be treated as valid primary-key
candidates for their respective datasets:

- `customer_id` → Customers
- `order_id` → Orders
- `product_id` → Products
- `seller_id` → Sellers
- `mql_id` → Marketing Qualified Leads
- `mql_id` → Closed Deals

These keys will be used during foreign-key and relationship validation.

## 7. Foreign Key Validation

Foreign-key validation checks whether identifiers in child datasets
actually exist in their corresponding parent datasets.

The purpose is to identify orphan records that cannot be matched to
their expected parent records.

Expected relationships include:

- Orders → Customers
- Order Items → Orders
- Order Items → Products
- Order Items → Sellers
- Payments → Orders
- Reviews → Orders
- Closed Deals → Marketing Qualified Leads

The validation results will be compared with the previously designed
data model.

In [20]:
# Load required datasets

customers = pd.read_csv(
    RAW_DATA_PATH / "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    RAW_DATA_PATH / "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    RAW_DATA_PATH / "olist_order_items_dataset.csv"
)

products = pd.read_csv(
    RAW_DATA_PATH / "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    RAW_DATA_PATH / "olist_sellers_dataset.csv"
)

payments = pd.read_csv(
    RAW_DATA_PATH / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    RAW_DATA_PATH / "olist_order_reviews_dataset.csv"
)

mql = pd.read_csv(
    RAW_DATA_PATH / "olist_marketing_qualified_leads_dataset.csv"
)

closed_deals = pd.read_csv(
    RAW_DATA_PATH / "olist_closed_deals_dataset.csv"
)

In [21]:
def validate_foreign_key(
    child_df,
    child_key,
    parent_df,
    parent_key,
    child_name,
    parent_name
):
    """
    Validate whether every child foreign-key value
    exists in the parent primary-key column.
    """

    child_values = child_df[child_key].dropna()
    parent_values = set(parent_df[parent_key].dropna())

    orphan_mask = ~child_values.isin(parent_values)

    orphan_count = orphan_mask.sum()
    child_count = len(child_values)

    match_count = child_count - orphan_count

    match_rate = (
        match_count / child_count * 100
        if child_count > 0
        else 0
    )

    return {
        "child_dataset": child_name,
        "child_key": child_key,
        "parent_dataset": parent_name,
        "parent_key": parent_key,
        "child_records": child_count,
        "matched_records": match_count,
        "orphan_records": orphan_count,
        "match_rate_%": round(match_rate, 2)
    }

In [22]:
fk_results = []

fk_results.append(
    validate_foreign_key(
        orders,
        "customer_id",
        customers,
        "customer_id",
        "orders",
        "customers"
    )
)

fk_results.append(
    validate_foreign_key(
        order_items,
        "order_id",
        orders,
        "order_id",
        "order_items",
        "orders"
    )
)

fk_results.append(
    validate_foreign_key(
        order_items,
        "product_id",
        products,
        "product_id",
        "order_items",
        "products"
    )
)

fk_results.append(
    validate_foreign_key(
        order_items,
        "seller_id",
        sellers,
        "seller_id",
        "order_items",
        "sellers"
    )
)

fk_results.append(
    validate_foreign_key(
        payments,
        "order_id",
        orders,
        "order_id",
        "payments",
        "orders"
    )
)

fk_results.append(
    validate_foreign_key(
        reviews,
        "order_id",
        orders,
        "order_id",
        "reviews",
        "orders"
    )
)

fk_results.append(
    validate_foreign_key(
        closed_deals,
        "mql_id",
        mql,
        "mql_id",
        "closed_deals",
        "marketing_qualified_leads"
    )
)

fk_validation_df = pd.DataFrame(fk_results)

fk_validation_df

,child_dataset,child_key,parent_dataset,parent_key,child_records,matched_records,orphan_records,match_rate_%
0,orders,customer_id,customers,customer_id,99441,99441,0,100.0
1,order_items,order_id,orders,order_id,112650,112650,0,100.0
2,order_items,product_id,products,product_id,112650,112650,0,100.0
3,order_items,seller_id,sellers,seller_id,112650,112650,0,100.0
4,payments,order_id,orders,order_id,103886,103886,0,100.0
5,reviews,order_id,orders,order_id,99224,99224,0,100.0
6,closed_deals,mql_id,marketing_qualified_leads,mql_id,842,842,0,100.0


## Foreign Key Validation Findings

All seven tested foreign-key relationships achieved a 100% match rate
with zero orphan records.

Validated relationships:

- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `payments.order_id` → `orders.order_id`
- `reviews.order_id` → `orders.order_id`
- `closed_deals.mql_id` → `marketing_qualified_leads.mql_id`

These results indicate that all tested foreign-key values have
corresponding parent records.

This validates the referential integrity of the tested relationships.

However, this does not imply that the datasets are completely free
from data-quality issues. Additional checks are required for missing
values, duplicates, data types, date validity, categorical consistency,
and business rules.

In [23]:
fk_validation_df.to_csv(
    REPORT_PATH / "foreign_key_validation_report.csv",
    index=False
)

print("Foreign-key validation report saved successfully.")

Foreign-key validation report saved successfully.


## 8. Relationship Cardinality Analysis

Foreign-key validation confirmed that the tested child records can be
matched to their corresponding parent records.

The next step is to analyze relationship cardinality.

This determines how many child records are associated with each parent
record and helps validate whether the relationships designed in the ERD
are actually supported by the data.

Relationships to investigate:

- Customer → Orders
- Order → Order Items
- Order → Payments
- Order → Reviews
- Product → Order Items
- Seller → Order Items

In [24]:
customer_order_counts = (
    orders
    .groupby("customer_id")
    .size()
    .reset_index(name="order_count")
)

customer_order_counts.head()

,customer_id,order_count
0,00012a2ce6f8dcda20d059ce98491703,1
1,000161a058600d5901f007fab4c27140,1
2,0001fd6190edaaf884bcaf3d49edf079,1
3,0002414f95344307404f0ace7a26f1d5,1
4,000379cdec625522490c315e70c7a9fb,1


In [25]:
customer_order_counts["order_count"].describe()

count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: order_count, dtype: float64

In [26]:
customer_order_counts["order_count"].max()

np.int64(1)

In [27]:
order_item_counts = (
    order_items
    .groupby("order_id")
    .size()
    .reset_index(name="item_count")
)

order_item_counts["item_count"].describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

In [28]:
order_item_counts["item_count"].max()

np.int64(21)

In [29]:
order_payment_counts = (
    payments
    .groupby("order_id")
    .size()
    .reset_index(name="payment_count")
)

order_payment_counts["payment_count"].describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
Name: payment_count, dtype: float64

In [30]:
order_payment_counts["payment_count"].max()

np.int64(29)

In [31]:
order_review_counts = (
    reviews
    .groupby("order_id")
    .size()
    .reset_index(name="review_count")
)

order_review_counts["review_count"].describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: review_count, dtype: float64

In [32]:
order_review_counts["review_count"].max()

np.int64(3)

In [33]:
product_item_counts = (
    order_items
    .groupby("product_id")
    .size()
    .reset_index(name="times_sold")
)

product_item_counts["times_sold"].describe()

count    32951.000000
mean         3.418713
std         10.619709
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        527.000000
Name: times_sold, dtype: float64

In [34]:
product_item_counts["times_sold"].max()

np.int64(527)

In [35]:
seller_item_counts = (
    order_items
    .groupby("seller_id")
    .size()
    .reset_index(name="items_sold")
)

seller_item_counts["items_sold"].describe()

count    3095.000000
mean       36.397415
std       119.193461
min         1.000000
25%         2.000000
50%         8.000000
75%        24.000000
max      2033.000000
Name: items_sold, dtype: float64

In [36]:
seller_item_counts["items_sold"].max()

np.int64(2033)

In [37]:
order_payment_counts["payment_count"].value_counts().sort_index()

payment_count
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [38]:
customer_unique_order_counts = (
    customers
    .merge(
        orders[["customer_id", "order_id"]],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="order_count")
)

customer_unique_order_counts["order_count"].describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: order_count, dtype: float64

In [39]:
customer_unique_order_counts["order_count"].value_counts().sort_index()

order_count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [40]:
customer_unique_order_counts["order_count"].max()

np.int64(17)

In [41]:
order_review_counts["review_count"].value_counts().sort_index()

review_count
1    98126
2      543
3        4
Name: count, dtype: int64

## Cardinality Analysis Findings

Cardinality analysis was performed to determine how many child records
can be associated with each parent record.

### Findings

- `customer_id` → `orders`: 1-to-1 within the available dataset.
- `customer_unique_id` → `orders`: 1-to-many, with a maximum of 17 orders
  associated with one unique customer.
- `orders` → `order_items`: 1-to-many, with up to 21 items per order.
- `orders` → `payments`: 1-to-many, with most orders having one payment
  record but some having multiple payment records.
- `orders` → `reviews`: 1-to-many in the observed data, with most orders
  having one review and a small number having multiple reviews.
- `products` → `order_items`: 1-to-many, with a maximum of 527 order-item
  records for one product.
- `sellers` → `order_items`: 1-to-many, with a maximum of 2,033 order-item
  records for one seller.

### Important Analytical Finding

`customer_id` and `customer_unique_id` represent different analytical
levels.

Each `customer_id` is associated with one order in the available data,
while `customer_unique_id` can be associated with multiple orders.

Therefore, `customer_unique_id` should be used when performing
customer-level analyses such as repeat purchasing, retention, customer
lifetime value, segmentation, and churn analysis.

Further investigation is required for orders containing multiple
payment or review records before determining whether these represent
legitimate business events or data-quality issues.

## 9. Missing Value Analysis

Missing-value analysis identifies incomplete records across all datasets.

The analysis will measure:

- Number of missing values
- Percentage of missing values
- Columns affected
- Datasets most affected

Missing values will not be automatically imputed or removed at this stage.

Each missing-value pattern will first be investigated to determine whether
the missingness represents:

- A legitimate business condition
- An unavailable attribute
- An incomplete transaction
- A data collection issue
- Or a potential data-quality problem

The appropriate treatment will be decided during the data-cleaning phase.

In [43]:
missing_results = []

for file in RAW_DATA_PATH.glob("*.csv"):

    df = pd.read_csv(file)

    for column in df.columns:

        null_count = df[column].isna().sum()
        null_percentage = (null_count / len(df)) * 100

        missing_results.append({
            "dataset": file.name,
            "column": column,
            "row_count": len(df),
            "null_count": null_count,
            "null_percentage": round(null_percentage, 2)
        })

missing_value_report = pd.DataFrame(missing_results)

missing_value_report

,dataset,column,row_count,null_count,null_percentage
0,olist_closed_deals_dataset.csv,mql_id,842,0,0.0
1,olist_closed_deals_dataset.csv,seller_id,842,0,0.0
2,olist_closed_deals_dataset.csv,sdr_id,842,0,0.0
3,olist_closed_deals_dataset.csv,sr_id,842,0,0.0
4,olist_closed_deals_dataset.csv,won_date,842,0,0.0
...,...,...,...,...,...
65,olist_sellers_dataset.csv,seller_zip_code_prefix,3095,0,0.0
66,olist_sellers_dataset.csv,seller_city,3095,0,0.0
67,olist_sellers_dataset.csv,seller_state,3095,0,0.0
68,product_category_name_translation.csv,product_category_name,71,0,0.0


In [44]:
missing_columns = (
    missing_value_report[
        missing_value_report["null_count"] > 0
    ]
    .sort_values(
        by="null_percentage",
        ascending=False
    )
)

missing_columns

,dataset,column,row_count,null_count,null_percentage
8,olist_closed_deals_dataset.csv,has_company,842,779,92.52
9,olist_closed_deals_dataset.csv,has_gtin,842,778,92.40
10,olist_closed_deals_dataset.csv,average_stock,842,776,92.16
12,olist_closed_deals_dataset.csv,declared_product_catalog_size,842,773,91.81
51,olist_order_reviews_dataset.csv,review_comment_title,99224,87656,88.34
52,olist_order_reviews_dataset.csv,review_comment_message,99224,58247,58.70
7,olist_closed_deals_dataset.csv,lead_behaviour_profile,842,177,21.02
34,olist_orders_dataset.csv,order_delivered_customer_date,99441,2965,2.98
59,olist_products_dataset.csv,product_photos_qty,32951,610,1.85
58,olist_products_dataset.csv,product_description_lenght,32951,610,1.85


In [45]:
missing_value_report.to_csv(
    REPORT_PATH / "missing_value_report.csv",
    index=False
)

print("Missing-value report saved successfully.")

Missing-value report saved successfully.


In [46]:
dataset_missing_summary = (
    missing_value_report
    .groupby("dataset")
    .agg(
        total_missing_values=("null_count", "sum"),
        columns_with_missing_values=(
            "null_count",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
    .sort_values(
        by="total_missing_values",
        ascending=False
    )
)

dataset_missing_summary

,dataset,total_missing_values,columns_with_missing_values
6,olist_order_reviews_dataset.csv,145903,2
7,olist_orders_dataset.csv,4908,3
0,olist_closed_deals_dataset.csv,3300,8
8,olist_products_dataset.csv,2448,8
3,olist_marketing_qualified_leads_dataset.csv,60,1
2,olist_geolocation_dataset.csv,0,0
1,olist_customers_dataset.csv,0,0
5,olist_order_payments_dataset.csv,0,0
4,olist_order_items_dataset.csv,0,0
9,olist_sellers_dataset.csv,0,0


In [47]:
missing_value_report["completeness_percentage"] = (
    100 - missing_value_report["null_percentage"]
)

missing_value_report.sort_values(
    by="completeness_percentage"
).head(20)

,dataset,column,row_count,null_count,null_percentage,completeness_percentage
8,olist_closed_deals_dataset.csv,has_company,842,779,92.52,7.48
9,olist_closed_deals_dataset.csv,has_gtin,842,778,92.40,7.60
10,olist_closed_deals_dataset.csv,average_stock,842,776,92.16,7.84
12,olist_closed_deals_dataset.csv,declared_product_catalog_size,842,773,91.81,8.19
51,olist_order_reviews_dataset.csv,review_comment_title,99224,87656,88.34,11.66
52,olist_order_reviews_dataset.csv,review_comment_message,99224,58247,58.70,41.30
7,olist_closed_deals_dataset.csv,lead_behaviour_profile,842,177,21.02,78.98
34,olist_orders_dataset.csv,order_delivered_customer_date,99441,2965,2.98,97.02
57,olist_products_dataset.csv,product_name_lenght,32951,610,1.85,98.15
58,olist_products_dataset.csv,product_description_lenght,32951,610,1.85,98.15


### 9.2 Missing Delivery Dates by Order Status

Delivery-related date fields contain missing values.

Before deciding how to handle these values, the relationship between
missing delivery dates and `order_status` will be investigated.

This will determine whether missing delivery dates represent legitimate
business conditions, such as cancelled or unavailable orders, rather than
data-quality errors.

In [48]:
delivery_missing_analysis = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_carrier_date=(
            "order_delivered_carrier_date",
            lambda x: x.isna().sum()
        ),
        missing_customer_delivery_date=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

delivery_missing_analysis

,order_status,total_orders,missing_carrier_date,missing_customer_delivery_date
0,approved,2,2,2
1,canceled,625,550,619
2,created,5,5,5
3,delivered,96478,2,8
4,invoiced,314,314,314
5,processing,301,301,301
6,shipped,1107,0,1107
7,unavailable,609,609,609


In [49]:
delivery_missing_analysis["carrier_missing_%"] = (
    delivery_missing_analysis["missing_carrier_date"]
    / delivery_missing_analysis["total_orders"]
    * 100
)

delivery_missing_analysis["customer_delivery_missing_%"] = (
    delivery_missing_analysis["missing_customer_delivery_date"]
    / delivery_missing_analysis["total_orders"]
    * 100
)

delivery_missing_analysis

,order_status,total_orders,missing_carrier_date,missing_customer_delivery_date,carrier_missing_%,customer_delivery_missing_%
0,approved,2,2,2,100.000000,100.000000
1,canceled,625,550,619,88.000000,99.040000
2,created,5,5,5,100.000000,100.000000
3,delivered,96478,2,8,0.002073,0.008292
4,invoiced,314,314,314,100.000000,100.000000
5,processing,301,301,301,100.000000,100.000000
6,shipped,1107,0,1107,0.000000,100.000000
7,unavailable,609,609,609,100.000000,100.000000


In [50]:
review_missing_analysis = (
    reviews.groupby("review_score")
    .agg(
        total_reviews=("review_id", "count"),
        missing_title=(
            "review_comment_title",
            lambda x: x.isna().sum()
        ),
        missing_message=(
            "review_comment_message",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

review_missing_analysis

,review_score,total_reviews,missing_title,missing_message
0,1,11424,9551,2679
1,2,3151,2673,1006
2,3,8179,7355,4622
3,4,19142,17407,13166
4,5,57328,50670,36774


In [51]:
review_missing_analysis["title_missing_%"] = (
    review_missing_analysis["missing_title"]
    / review_missing_analysis["total_reviews"]
    * 100
)

review_missing_analysis["message_missing_%"] = (
    review_missing_analysis["missing_message"]
    / review_missing_analysis["total_reviews"]
    * 100
)

review_missing_analysis

,review_score,total_reviews,missing_title,missing_message,title_missing_%,message_missing_%
0,1,11424,9551,2679,83.604692,23.450630
1,2,3151,2673,1006,84.830213,31.926373
2,3,8179,7355,4622,89.925419,56.510576
3,4,19142,17407,13166,90.936161,68.780692
4,5,57328,50670,36774,88.386129,64.146665


In [52]:
product_missing_analysis = products[
    products[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty"
        ]
    ].isna().any(axis=1)
]

product_missing_analysis[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].head(20)

,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty
105,NaN,NaN,NaN,NaN
128,NaN,NaN,NaN,NaN
145,NaN,NaN,NaN,NaN
154,NaN,NaN,NaN,NaN
197,NaN,NaN,NaN,NaN
244,NaN,NaN,NaN,NaN
294,NaN,NaN,NaN,NaN
299,NaN,NaN,NaN,NaN
347,NaN,NaN,NaN,NaN
428,NaN,NaN,NaN,NaN


In [53]:
product_missing_analysis[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().sum()

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

In [54]:
product_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

product_missing_mask = products[product_cols].isna()

product_missing_pattern = (
    product_missing_mask
    .value_counts()
    .reset_index(name="row_count")
)

product_missing_pattern

,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,row_count
0,False,False,False,False,32341
1,True,True,True,True,610


### 9.4 Final Missing Value Findings

The missing-value investigation identified several distinct patterns.

#### Order Data
Missing delivery dates are primarily associated with legitimate order
statuses where delivery has not occurred or cannot occur.

Therefore, delivery dates will not be automatically imputed.

A very small number of delivered orders have missing delivery dates and
will be treated as potential data-quality anomalies for further
investigation.

#### Review Data
Review scores are available even when customers do not provide written
comments.

Therefore, missing review titles and messages are considered valid
business conditions rather than automatically treated as data-quality
errors.

#### Product Data
Exactly 610 product records have all four of the following attributes
missing simultaneously:

- product_category_name
- product_name_lenght
- product_description_lenght
- product_photos_qty

The remaining 32,341 product records contain all four attributes.

This confirms that the missing values form a consistent record-level
pattern rather than independent missing values across the four columns.

#### Final Decision
Missing values will not be globally imputed or deleted.

Treatment will be determined later during data cleaning based on:

1. Business meaning
2. Analytical purpose
3. Relationship with other variables
4. Impact on downstream analysis and machine-learning models

## 10. Duplicate & Record Uniqueness Analysis

Duplicate analysis identifies repeated records and verifies whether
candidate keys maintain the expected uniqueness.

Two types of duplication will be investigated:

1. Exact row duplicates
2. Key-level duplication

Repeated foreign keys will not automatically be treated as duplicates,
because one-to-many business relationships naturally contain repeated
identifiers.

In [55]:
duplicate_results = []

for file in RAW_DATA_PATH.glob("*.csv"):

    df = pd.read_csv(file)

    duplicate_count = df.duplicated().sum()

    duplicate_results.append({
        "dataset": file.name,
        "row_count": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            duplicate_count / len(df) * 100, 2
        )
    })

duplicate_report = pd.DataFrame(duplicate_results)

duplicate_report

,dataset,row_count,duplicate_rows,duplicate_percentage
0,olist_closed_deals_dataset.csv,842,0,0.00
1,olist_customers_dataset.csv,99441,0,0.00
2,olist_geolocation_dataset.csv,1000163,261831,26.18
3,olist_marketing_qualified_leads_dataset.csv,8000,0,0.00
4,olist_orders_dataset.csv,99441,0,0.00
5,olist_order_items_dataset.csv,112650,0,0.00
6,olist_order_payments_dataset.csv,103886,0,0.00
7,olist_order_reviews_dataset.csv,99224,0,0.00
8,olist_products_dataset.csv,32951,0,0.00
9,olist_sellers_dataset.csv,3095,0,0.00


In [56]:
duplicate_report.to_csv(
    REPORT_PATH / "duplicate_report.csv",
    index=False
)

print("Duplicate report saved successfully.")

Duplicate report saved successfully.


In [57]:
candidate_keys = {
    "olist_customers_dataset.csv": "customer_id",
    "olist_orders_dataset.csv": "order_id",
    "olist_products_dataset.csv": "product_id",
    "olist_sellers_dataset.csv": "seller_id",
    "olist_marketing_qualified_leads_dataset.csv": "mql_id",
    "olist_closed_deals_dataset.csv": "mql_id"
}

key_results = []

for dataset, key in candidate_keys.items():

    file = RAW_DATA_PATH / dataset
    df = pd.read_csv(file)

    row_count = len(df)
    null_count = df[key].isna().sum()
    unique_count = df[key].nunique()
    duplicate_count = row_count - unique_count

    key_results.append({
        "dataset": dataset,
        "key": key,
        "row_count": row_count,
        "null_count": null_count,
        "unique_count": unique_count,
        "duplicate_count": duplicate_count,
        "is_unique": (
            null_count == 0
            and duplicate_count == 0
        )
    })

key_validation = pd.DataFrame(key_results)

key_validation

,dataset,key,row_count,null_count,unique_count,duplicate_count,is_unique
0,olist_customers_dataset.csv,customer_id,99441,0,99441,0,True
1,olist_orders_dataset.csv,order_id,99441,0,99441,0,True
2,olist_products_dataset.csv,product_id,32951,0,32951,0,True
3,olist_sellers_dataset.csv,seller_id,3095,0,3095,0,True
4,olist_marketing_qualified_leads_dataset.csv,mql_id,8000,0,8000,0,True
5,olist_closed_deals_dataset.csv,mql_id,842,0,842,0,True


In [58]:
customers = pd.read_csv(
    RAW_DATA_PATH / "olist_customers_dataset.csv"
)

customer_id_unique = customers["customer_id"].nunique()
customer_unique_id_unique = customers["customer_unique_id"].nunique()

print("Unique customer_id:", customer_id_unique)
print("Unique customer_unique_id:", customer_unique_id_unique)
print(
    "Repeated customer_unique_id records:",
    len(customers) - customer_unique_id_unique
)

Unique customer_id: 99441
Unique customer_unique_id: 96096
Repeated customer_unique_id records: 3345


In [59]:
customer_frequency = (
    customers
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="customer_id_count")
    .sort_values(
        by="customer_id_count",
        ascending=False
    )
)

customer_frequency.head(20)

,customer_unique_id,customer_id_count
52973,8d50f5eadf50201ccdcedfb9e2ac8455,17
23472,3e43e6105506432c953e165fb2acf44c,9
10354,1b6c7548a2a1f9037c1fd3ddfed95f33,7
37797,6469f99c1f9dfae7733b25662e7f1782,7
76082,ca77025e7201e3b30c44b472ff346268,7
27043,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
7175,12f5d6e1cbf93dafd9dcc19095df0b3d,6
37585,63cfc61cee11cbe306bff5857d00bfe4,6
82883,dc813062e0fc23409cd255f7f53c7074,6
83540,de34b16117594161a6a89c50b289d35a,6


In [60]:
customer_frequency["customer_id_count"].value_counts().sort_index()

customer_id_count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [61]:
order_items = pd.read_csv(
    RAW_DATA_PATH / "olist_order_items_dataset.csv"
)

order_item_frequency = (
    order_items
    .groupby("order_id")
    .size()
    .reset_index(name="item_count")
)

order_item_frequency["item_count"].value_counts().sort_index()

item_count
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [62]:
reviews = pd.read_csv(
    RAW_DATA_PATH / "olist_order_reviews_dataset.csv"
)

review_frequency = (
    reviews
    .groupby("order_id")
    .size()
    .reset_index(name="review_count")
)

review_frequency[
    review_frequency["review_count"] > 1
].sort_values(
    by="review_count",
    ascending=False
).head(20)

,order_id,review_count
1455,03c939fd7fd3b38f8485a0f95798f1f6,3
54489,8e17072ec97ce29f0e1f111e598b0c85,3
77319,c88b1d1b157a9999ce368f218a407141,3
86232,df56136b8031ecd28e200bb18e6ddb2e,3
835,02355020fd0a40a0d56df9f6ff060413,2
985,029863af4b968de1e5d6a82782e662f5,2
1092,02e0b68852217f5715fb9cc885829454,2
1103,02e723e8edb4a123d414f56cc9c4665e,2
1272,03515a836bb855b03f7df9dee520a8fc,2
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2


In [63]:
customer_mapping = (
    customers
    .groupby("customer_unique_id")
    .agg(
        customer_id_count=("customer_id", "nunique"),
        customer_city_count=("customer_city", "nunique"),
        customer_state_count=("customer_state", "nunique"),
        customer_zip_count=("customer_zip_code_prefix", "nunique")
    )
    .reset_index()
)

customer_mapping.sort_values(
    "customer_id_count",
    ascending=False
).head(20)

,customer_unique_id,customer_id_count,customer_city_count,customer_state_count,customer_zip_count
52973,8d50f5eadf50201ccdcedfb9e2ac8455,17,1,1,1
23472,3e43e6105506432c953e165fb2acf44c,9,1,1,3
10354,1b6c7548a2a1f9037c1fd3ddfed95f33,7,1,1,1
37797,6469f99c1f9dfae7733b25662e7f1782,7,1,1,1
76082,ca77025e7201e3b30c44b472ff346268,7,1,1,1
27043,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,1,1,2
7175,12f5d6e1cbf93dafd9dcc19095df0b3d,6,1,1,1
37585,63cfc61cee11cbe306bff5857d00bfe4,6,1,1,1
82883,dc813062e0fc23409cd255f7f53c7074,6,1,1,1
83540,de34b16117594161a6a89c50b289d35a,6,1,1,1


In [64]:
geolocation = pd.read_csv(
    RAW_DATA_PATH / "olist_geolocation_dataset.csv"
)

geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [65]:
geolocation.nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [66]:
geolocation.columns

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')

In [67]:
geolocation.groupby(
    ["geolocation_zip_code_prefix"]
).size().describe()

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64

In [68]:
geo_grain = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        row_count=("geolocation_zip_code_prefix", "size"),
        unique_lat=("geolocation_lat", "nunique"),
        unique_lng=("geolocation_lng", "nunique"),
        unique_city=("geolocation_city", "nunique"),
        unique_state=("geolocation_state", "nunique")
    )
    .reset_index()
)

geo_grain.head(20)

,geolocation_zip_code_prefix,row_count,unique_lat,unique_lng,unique_city,unique_state
0,1001,26,10,10,2,1
1,1002,13,6,6,2,1
2,1003,17,10,10,2,1
3,1004,22,14,14,2,1
4,1005,25,11,11,2,1
5,1006,9,7,7,2,1
6,1007,26,15,15,2,1
7,1008,16,12,12,2,1
8,1009,41,7,7,2,1
9,1010,18,5,5,2,1


In [69]:
geo_grain.describe()

,geolocation_zip_code_prefix,row_count,unique_lat,unique_lng,unique_city,unique_state
count,19015.000000,19015.000000,19015.000000,19015.000000,19015.000000,19015.000000
mean,42711.591901,52.598633,37.828031,37.835656,1.467631,1.000421
std,30905.051745,72.057907,49.929342,49.944594,0.538505,0.020508
min,1001.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,12721.500000,10.000000,8.000000,8.000000,1.000000,1.000000
50%,38240.000000,29.000000,22.000000,22.000000,1.000000,1.000000
75%,70656.500000,66.500000,48.000000,48.000000,2.000000,1.000000
max,99990.000000,1146.000000,746.000000,745.000000,5.000000,2.000000


In [70]:
geo_duplicates = geolocation[
    geolocation.duplicated(keep=False)
].sort_values(
    by=list(geolocation.columns)
)

geo_duplicates.head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


In [71]:
geo_conflicts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        unique_lat=("geolocation_lat", "nunique"),
        unique_lng=("geolocation_lng", "nunique"),
        unique_city=("geolocation_city", "nunique"),
        unique_state=("geolocation_state", "nunique")
    )
    .reset_index()
)

geo_conflicts[
    (geo_conflicts["unique_city"] > 1) |
    (geo_conflicts["unique_state"] > 1)
].sort_values(
    "unique_city",
    ascending=False
).head(20)

,geolocation_zip_code_prefix,unique_lat,unique_lng,unique_city,unique_state
5064,13454,118,118,5,1
5067,13457,42,42,5,1
4986,13318,41,41,5,1
5065,13455,71,71,5,1
5966,17970,14,14,5,1
10239,42850,178,178,5,1
3443,6900,380,380,5,1
7726,28950,274,274,5,1
15821,78290,26,26,5,1
5060,13450,133,133,4,1


In [72]:
geo_conflicts[
    (geo_conflicts["unique_city"] > 1) |
    (geo_conflicts["unique_state"] > 1)
].sort_values(
    "unique_city",
    ascending=False
).head(20)

,geolocation_zip_code_prefix,unique_lat,unique_lng,unique_city,unique_state
5064,13454,118,118,5,1
5067,13457,42,42,5,1
4986,13318,41,41,5,1
5065,13455,71,71,5,1
5966,17970,14,14,5,1
10239,42850,178,178,5,1
3443,6900,380,380,5,1
7726,28950,274,274,5,1
15821,78290,26,26,5,1
5060,13450,133,133,4,1


## 10. Duplicate & Record Uniqueness Findings

### Exact Duplicate Analysis

Most datasets contain no exact duplicate rows.

The geolocation dataset is the exception, containing 261,831 exact
duplicate rows out of 1,000,163 records (26.18%).

### Candidate Key Validation

The following identifiers were confirmed to be unique and non-null:

- customer_id
- order_id
- product_id
- seller_id
- marketing qualified lead mql_id
- closed deal mql_id

These can be treated as candidate primary keys at their respective
table grains.

### Customer Identity

customer_id is unique at the customer-record level, while
customer_unique_id is not unique.

Multiple customer_id records can belong to the same
customer_unique_id. Therefore, customer_unique_id will be used as the
primary customer identity for customer-level behavioral analysis,
while customer_id will remain useful for record/order relationships.

### Geolocation

The geolocation dataset contains substantial exact duplication.
However, a ZIP-code prefix can contain multiple geographic coordinates
and multiple city observations.

Therefore, the raw geolocation dataset will not be reduced to one row
per ZIP-code prefix.

Exact duplicates may be removed during a later transformation if the
resulting table's analytical grain requires unique geographic
observations. The raw source will remain preserved.

### Key Principle

Duplicates are not automatically data-quality errors.

Before removing a duplicate, the intended grain of the dataset and the
business meaning of the repeated record must be understood.

## 11. Numerical Distribution & Outlier Analysis

This step examines numerical variables to understand their distributions,
central tendency, variability, skewness, and potential extreme values.

Outliers will not automatically be removed. Each extreme value will be
evaluated using business context before any cleaning decision is made.

The analysis will focus primarily on variables that can influence:

- Revenue
- Profit
- Orders
- Product performance
- Customer behavior
- Payments
- Delivery performance
- Inventory
- Machine-learning models

In [73]:
numeric_results = []

for file in RAW_DATA_PATH.glob("*.csv"):

    df = pd.read_csv(file)

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    for col in numeric_cols:

        numeric_results.append({
            "dataset": file.name,
            "column": col,
            "data_type": str(df[col].dtype),
            "non_null": df[col].notna().sum(),
            "unique_values": df[col].nunique()
        })

numeric_inventory = pd.DataFrame(numeric_results)

numeric_inventory

,dataset,column,data_type,non_null,unique_values
0,olist_closed_deals_dataset.csv,declared_product_catalog_size,float64,69,33
1,olist_closed_deals_dataset.csv,declared_monthly_revenue,float64,842,27
2,olist_customers_dataset.csv,customer_zip_code_prefix,int64,99441,14994
3,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,int64,1000163,19015
4,olist_geolocation_dataset.csv,geolocation_lat,float64,1000163,717360
5,olist_geolocation_dataset.csv,geolocation_lng,float64,1000163,717613
6,olist_order_items_dataset.csv,order_item_id,int64,112650,21
7,olist_order_items_dataset.csv,price,float64,112650,5968
8,olist_order_items_dataset.csv,freight_value,float64,112650,6999
9,olist_order_payments_dataset.csv,payment_sequential,int64,103886,29


In [74]:
distribution_results = []

for file in RAW_DATA_PATH.glob("*.csv"):

    df = pd.read_csv(file)

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    for col in numeric_cols:

        series = df[col].dropna()

        if len(series) == 0:
            continue

        distribution_results.append({
            "dataset": file.name,
            "column": col,
            "count": len(series),
            "min": series.min(),
            "q1": series.quantile(0.25),
            "median": series.median(),
            "mean": series.mean(),
            "q3": series.quantile(0.75),
            "max": series.max(),
            "std": series.std(),
            "skewness": series.skew()
        })

distribution_profile = pd.DataFrame(
    distribution_results
)

distribution_profile

,dataset,column,count,min,q1,median,mean,q3,max,std,skewness
0,olist_closed_deals_dataset.csv,declared_product_catalog_size,69,1.000000,30.000000,100.000000,233.028986,300.000000,2.000000e+03,3.523806e+02,2.731955
1,olist_closed_deals_dataset.csv,declared_monthly_revenue,842,0.000000,0.000000,0.000000,73377.679335,0.000000,5.000000e+07,1.744799e+06,28.036956
2,olist_customers_dataset.csv,customer_zip_code_prefix,99441,1003.000000,11347.000000,24416.000000,35137.474583,58900.000000,9.999000e+04,2.979794e+04,0.779025
3,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,1000163,1001.000000,11075.000000,26530.000000,36574.166466,63504.000000,9.999000e+04,3.054934e+04,0.694494
4,olist_geolocation_dataset.csv,geolocation_lat,1000163,-36.605374,-23.603546,-22.919377,-21.176153,-19.979620,4.506593e+01,5.715866e+00,1.565147
5,olist_geolocation_dataset.csv,geolocation_lng,1000163,-101.466766,-48.573172,-46.637879,-46.390541,-43.767709,1.211054e+02,4.269748e+00,-0.102417
6,olist_order_items_dataset.csv,order_item_id,112650,1.000000,1.000000,1.000000,1.197834,1.000000,2.100000e+01,7.051240e-01,7.580356
7,olist_order_items_dataset.csv,price,112650,0.850000,39.900000,74.990000,120.653739,134.900000,6.735000e+03,1.836339e+02,7.923208
8,olist_order_items_dataset.csv,freight_value,112650,0.000000,13.080000,16.260000,19.990320,21.150000,4.096800e+02,1.580641e+01,5.639870
9,olist_order_payments_dataset.csv,payment_sequential,103886,1.000000,1.000000,1.000000,1.092679,1.000000,2.900000e+01,7.065838e-01,16.180065


In [75]:
distribution_profile.to_csv(
    REPORT_PATH / "numerical_distribution_profile.csv",
    index=False
)

print("Numerical distribution profile saved.")

Numerical distribution profile saved.


In [76]:
outlier_results = []

for file in RAW_DATA_PATH.glob("*.csv"):

    df = pd.read_csv(file)

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    for col in numeric_cols:

        series = df[col].dropna()

        if len(series) < 5:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)

        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (series < lower_bound) |
            (series > upper_bound)
        ).sum()

        outlier_results.append({
            "dataset": file.name,
            "column": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": outlier_count,
            "outlier_percentage": round(
                outlier_count / len(series) * 100,
                2
            )
        })

outlier_profile = pd.DataFrame(
    outlier_results
)

outlier_profile.sort_values(
    "outlier_percentage",
    ascending=False
)

,dataset,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
4,olist_geolocation_dataset.csv,geolocation_lat,-23.603546,-19.979620,3.623925,-29.039433,-14.543733,168240,16.82
12,olist_order_reviews_dataset.csv,review_score,4.000000,5.000000,1.000000,2.500000,6.500000,14575,14.69
16,olist_products_dataset.csv,product_weight_g,300.000000,1900.000000,1600.000000,-2100.000000,4300.000000,4551,13.81
6,olist_order_items_dataset.csv,order_item_id,1.000000,1.000000,0.000000,1.000000,1.000000,13984,12.41
8,olist_order_items_dataset.csv,freight_value,13.080000,21.150000,8.070000,0.975000,33.255000,12134,10.77
0,olist_closed_deals_dataset.csv,declared_product_catalog_size,30.000000,300.000000,270.000000,-375.000000,705.000000,6,8.70
11,olist_order_payments_dataset.csv,payment_value,56.790000,171.837500,115.047500,-115.781250,344.408750,7981,7.68
7,olist_order_items_dataset.csv,price,39.900000,134.900000,95.000000,-102.600000,277.400000,8427,7.48
14,olist_products_dataset.csv,product_description_lenght,339.000000,972.000000,633.000000,-610.500000,1921.500000,2078,6.43
10,olist_order_payments_dataset.csv,payment_installments,1.000000,4.000000,3.000000,-3.500000,8.500000,6313,6.08


In [77]:
outlier_profile[
    outlier_profile["outlier_count"] > 0
].sort_values(
    "outlier_percentage",
    ascending=False
).head(30)

,dataset,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
4,olist_geolocation_dataset.csv,geolocation_lat,-23.603546,-19.979620,3.623925,-29.039433,-14.543733,168240,16.82
12,olist_order_reviews_dataset.csv,review_score,4.000000,5.000000,1.000000,2.500000,6.500000,14575,14.69
16,olist_products_dataset.csv,product_weight_g,300.000000,1900.000000,1600.000000,-2100.000000,4300.000000,4551,13.81
6,olist_order_items_dataset.csv,order_item_id,1.000000,1.000000,0.000000,1.000000,1.000000,13984,12.41
8,olist_order_items_dataset.csv,freight_value,13.080000,21.150000,8.070000,0.975000,33.255000,12134,10.77
0,olist_closed_deals_dataset.csv,declared_product_catalog_size,30.000000,300.000000,270.000000,-375.000000,705.000000,6,8.70
11,olist_order_payments_dataset.csv,payment_value,56.790000,171.837500,115.047500,-115.781250,344.408750,7981,7.68
7,olist_order_items_dataset.csv,price,39.900000,134.900000,95.000000,-102.600000,277.400000,8427,7.48
14,olist_products_dataset.csv,product_description_lenght,339.000000,972.000000,633.000000,-610.500000,1921.500000,2078,6.43
10,olist_order_payments_dataset.csv,payment_installments,1.000000,4.000000,3.000000,-3.500000,8.500000,6313,6.08


In [78]:
distribution_profile.sort_values(
    "skewness",
    ascending=False
).head(30)

,dataset,column,count,min,q1,median,mean,q3,max,std,skewness
1,olist_closed_deals_dataset.csv,declared_monthly_revenue,842,0.000000,0.000000,0.000000,73377.679335,0.000000,5.000000e+07,1.744799e+06,28.036956
9,olist_order_payments_dataset.csv,payment_sequential,103886,1.000000,1.000000,1.000000,1.092679,1.000000,2.900000e+01,7.065838e-01,16.180065
11,olist_order_payments_dataset.csv,payment_value,103886,0.000000,56.790000,100.000000,154.100380,171.837500,1.366408e+04,2.174941e+02,9.254010
7,olist_order_items_dataset.csv,price,112650,0.850000,39.900000,74.990000,120.653739,134.900000,6.735000e+03,1.836339e+02,7.923208
6,olist_order_items_dataset.csv,order_item_id,112650,1.000000,1.000000,1.000000,1.197834,1.000000,2.100000e+01,7.051240e-01,7.580356
8,olist_order_items_dataset.csv,freight_value,112650,0.000000,13.080000,16.260000,19.990320,21.150000,4.096800e+02,1.580641e+01,5.639870
16,olist_products_dataset.csv,product_weight_g,32949,0.000000,300.000000,700.000000,2276.472488,1900.000000,4.042500e+04,4.282039e+03,3.604860
0,olist_closed_deals_dataset.csv,declared_product_catalog_size,69,1.000000,30.000000,100.000000,233.028986,300.000000,2.000000e+03,3.523806e+02,2.731955
15,olist_products_dataset.csv,product_photos_qty,32341,1.000000,1.000000,1.000000,2.188986,3.000000,2.000000e+01,1.736766e+00,2.193409
18,olist_products_dataset.csv,product_height_cm,32949,2.000000,8.000000,13.000000,16.937661,21.000000,1.050000e+02,1.363755e+01,2.140061


## Step 11 Findings — Numerical Distribution & Outlier Analysis

The numerical profiling identified several highly skewed variables and
statistically unusual observations.

Important findings:

- `declared_monthly_revenue` is extremely right-skewed, with a median of 0
  and a maximum of 50,000,000.
- `payment_value`, `price`, and `freight_value` show strong right-skewed
  distributions, which can be expected in e-commerce transactions.
- `product_weight_g` and product dimensions contain legitimate-looking
  extreme values that may represent larger/heavier products.
- `review_score` contains statistical outliers according to the IQR method,
  but these values represent legitimate customer ratings and must not be
  removed.
- `order_item_id` and `payment_sequential` are structural variables and
  should not be treated as business measures for outlier removal.
- Geographic latitude and longitude should not be cleaned using a simple
  IQR-based outlier rule because they represent spatial data.

### Decision

No numerical values will be removed at this stage.

Statistical outliers will be retained unless further business or data-quality
investigation demonstrates that they are erroneous.

Potentially problematic variables, especially `declared_monthly_revenue`,
will be investigated before being used for analytical or machine-learning
purposes.

In [81]:
# Step 12: Categorical & Value Consistency

categorical_summary = []

for file in csv_files:
    df = pd.read_csv(file)

    categorical_cols = df.select_dtypes(
        include=["str", "object", "category"]
    ).columns

    for col in categorical_cols:
        categorical_summary.append({
            "dataset": file.name,
            "column": col,
            "unique_values": df[col].nunique(dropna=True),
            "missing_values": df[col].isna().sum()
        })

categorical_summary_df = pd.DataFrame(categorical_summary)

categorical_summary_df

,dataset,column,unique_values,missing_values
0,olist_closed_deals_dataset.csv,mql_id,842,0
1,olist_closed_deals_dataset.csv,seller_id,842,0
2,olist_closed_deals_dataset.csv,sdr_id,32,0
3,olist_closed_deals_dataset.csv,sr_id,22,0
4,olist_closed_deals_dataset.csv,won_date,824,0
5,olist_closed_deals_dataset.csv,business_segment,33,1
6,olist_closed_deals_dataset.csv,lead_type,8,6
7,olist_closed_deals_dataset.csv,lead_behaviour_profile,9,177
8,olist_closed_deals_dataset.csv,has_company,2,779
9,olist_closed_deals_dataset.csv,has_gtin,2,778


## Step 13: Cross-Field & Business Rule Validation

This step validates relationships and logical business rules across multiple fields and datasets.

The objective is to identify records that are structurally valid but logically inconsistent.

Checks include:
- Order-to-customer consistency
- Order status vs delivery dates
- Order timeline consistency
- Review score validity
- Product dimension and weight validity
- Payment value validity
- Business-rule violations relevant to ShopSphere

In [82]:
orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")

order_customer_check = (
    orders.groupby("order_id")["customer_id"]
    .nunique()
    .reset_index(name="unique_customer_count")
)

order_customer_check[order_customer_check["unique_customer_count"] > 1]

,order_id,unique_customer_count


In [83]:
orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"],
    errors="coerce"
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"],
    errors="coerce"
)

delivered_check = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

print("Delivered orders with missing delivery dates:", len(delivered_check))

Delivered orders with missing delivery dates: 9


In [84]:
non_delivered_with_delivery_date = orders[
    (orders["order_status"] != "delivered") &
    (orders["order_delivered_customer_date"].notna())
]

print(
    "Non-delivered orders with customer delivery date:",
    len(non_delivered_with_delivery_date)
)

Non-delivered orders with customer delivery date: 6


In [85]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

In [86]:
invalid_delivery_timeline = orders[
    (
        orders["order_delivered_customer_date"].notna()
    ) &
    (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    )
]

print(
    "Orders where customer delivery date is before purchase:",
    len(invalid_delivery_timeline)
)

Orders where customer delivery date is before purchase: 0


In [87]:
invalid_carrier_timeline = orders[
    (
        orders["order_delivered_carrier_date"].notna()
    ) &
    (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    )
]

print(
    "Orders where carrier date is before purchase:",
    len(invalid_carrier_timeline)
)

Orders where carrier date is before purchase: 166


In [88]:
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)

invalid_estimated_date = orders[
    orders["order_estimated_delivery_date"]
    < orders["order_purchase_timestamp"]
]

print(
    "Orders where estimated delivery is before purchase:",
    len(invalid_estimated_date)
)

Orders where estimated delivery is before purchase: 0


In [89]:
reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")

invalid_review_scores = reviews[
    ~reviews["review_score"].between(1, 5)
]

print("Invalid review scores:", len(invalid_review_scores))

Invalid review scores: 0


In [90]:
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")

physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

invalid_physical_values = {}

for col in physical_columns:
    invalid_physical_values[col] = (
        products[col].lt(0).sum()
    )

invalid_physical_values

{'product_weight_g': np.int64(0),
 'product_length_cm': np.int64(0),
 'product_height_cm': np.int64(0),
 'product_width_cm': np.int64(0)}

In [91]:
payments = pd.read_csv(
    RAW_DATA_PATH / "olist_order_payments_dataset.csv"
)

invalid_payment_values = payments[
    payments["payment_value"] < 0
]

print("Negative payment values:", len(invalid_payment_values))

Negative payment values: 0


In [92]:
zero_installments = payments[
    payments["payment_installments"] == 0
]

print("Payments with zero installments:", len(zero_installments))

Payments with zero installments: 2


In [93]:
products = pd.read_csv(
    RAW_DATA_PATH / "olist_products_dataset.csv"
)

translation = pd.read_csv(
    RAW_DATA_PATH / "product_category_name_translation.csv"
)

product_categories = set(
    products["product_category_name"]
    .dropna()
    .unique()
)

translation_categories = set(
    translation["product_category_name"]
    .dropna()
    .unique()
)

categories_without_translation = (
    product_categories - translation_categories
)

categories_without_translation

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

In [94]:
customers = pd.read_csv(
    RAW_DATA_PATH / "olist_customers_dataset.csv"
)

valid_customer_ids = set(customers["customer_id"])

invalid_customer_orders = orders[
    ~orders["customer_id"].isin(valid_customer_ids)
]

print(
    "Orders with invalid customer IDs:",
    len(invalid_customer_orders)
)

Orders with invalid customer IDs: 0


In [98]:
# ============================================================
# STEP 13: BUSINESS RULE & LOGICAL CONSISTENCY VALIDATION
# Final Summary
# ============================================================

validation_results = {
    "Delivered orders with missing delivery dates": {
        "count": 9,
        "status": "REVIEW"
    },

    "Non-delivered orders with customer delivery date": {
        "count": 6,
        "status": "REVIEW"
    },

    "Orders where customer delivery date is before purchase": {
        "count": 0,
        "status": "PASS"
    },

    "Orders where carrier date is before purchase": {
        "count": 166,
        "status": "REVIEW"
    },

    "Orders where estimated delivery is before purchase": {
        "count": 0,
        "status": "PASS"
    },

    "Invalid review scores": {
        "count": 0,
        "status": "PASS"
    },

    "Invalid product dimensions": {
        "count": 0,
        "status": "PASS"
    },

    "Negative payment values": {
        "count": 0,
        "status": "PASS"
    },

    "Payments with zero installments": {
        "count": 2,
        "status": "REVIEW"
    },

    "Untranslated product categories": {
        "count": 2,
        "status": "REVIEW"
    },

    "Orders with invalid customer IDs": {
        "count": 0,
        "status": "PASS"
    }
}

# Convert results into a DataFrame
business_rule_summary = pd.DataFrame([
    {
        "validation_check": check,
        "issue_count": result["count"],
        "status": result["status"]
    }
    for check, result in validation_results.items()
])

# Display final summary
display(business_rule_summary)

# Overall status
review_count = (business_rule_summary["status"] == "REVIEW").sum()
pass_count = (business_rule_summary["status"] == "PASS").sum()

print("\n" + "=" * 60)
print("STEP 13 FINAL VALIDATION SUMMARY")
print("=" * 60)

print(f"PASS checks   : {pass_count}")
print(f"REVIEW checks : {review_count}")

if review_count == 0:
    print("Overall Status: PASS")
else:
    print("Overall Status: REVIEW REQUIRED")

print("=" * 60)

,validation_check,issue_count,status
0,Delivered orders with missing delivery dates,9,REVIEW
1,Non-delivered orders with customer delivery date,6,REVIEW
2,Orders where customer delivery date is before ...,0,PASS
3,Orders where carrier date is before purchase,166,REVIEW
4,Orders where estimated delivery is before purc...,0,PASS
5,Invalid review scores,0,PASS
6,Invalid product dimensions,0,PASS
7,Negative payment values,0,PASS
8,Payments with zero installments,2,REVIEW
9,Untranslated product categories,2,REVIEW



STEP 13 FINAL VALIDATION SUMMARY
PASS checks   : 6
REVIEW checks : 5
Overall Status: REVIEW REQUIRED


Conclusion: Business-rule validation identified several records requiring review, primarily related to missing delivery dates, inconsistent order timestamps, zero payment installments, and untranslated product categories. No invalid review scores, negative payment values, invalid product dimensions, or invalid customer references were detected.

# Step 14: Final Data Profiling & Quality Summary

## Objective

Consolidate the findings from all data profiling and validation checks performed in Phase 5.

The objective is to determine:
- Dataset structure and coverage
- Data completeness
- Key uniqueness
- Referential integrity
- Duplicate records
- Distribution and outlier patterns
- Categorical consistency
- Business-rule violations
- Fields requiring review before data cleaning

## Overall Assessment

The Olist dataset contains 11 related datasets covering customers, orders,
order items, payments, reviews, products, sellers, geolocation, and marketing leads.

The profiling identified generally strong structural integrity and referential
relationships. However, several fields require attention during Phase 6,
particularly missing values, unusual distributions, duplicate geolocation
records, and business-rule exceptions.

## Key Findings

### Structural Quality
- 11 datasets identified.
- Primary/candidate keys were unique for the identified master entities.
- Referential integrity checks showed 100% matching records for the tested relationships.

### Completeness
- Customers, order items, payments, sellers, and geolocation datasets had no missing values.
- High missingness was identified in selected closed-deals and review-comment fields.
- Product attributes had approximately 1.85% missing values.
- Order delivery dates contain missing values that require business-context handling.

### Uniqueness & Duplicates
- No duplicate rows were identified in the major transactional datasets.
- Geolocation contains approximately 26.18% duplicate rows.
- `customer_unique_id` is intentionally non-unique because one customer can have multiple `customer_id` records.

### Distribution & Outliers
- Several numeric variables showed right-skewed distributions.
- High-value outliers were observed in payment values, item prices, freight values,
  product dimensions/weights, and declared revenue.
- Outliers will be investigated before deciding whether they represent genuine
  business values or data-quality issues.

### Categorical Consistency
- Most categorical fields contain valid and consistent values.
- Two product categories were identified without matching English translations.
- These will be reviewed during data cleaning.

### Business Rule Validation
- Most business rules passed.
- Exceptions requiring review include:
  - Delivered orders with missing delivery dates
  - Non-delivered orders with customer delivery dates
  - Carrier dates occurring before purchase dates
  - Payments with zero installments
  - Untranslated product categories

## Phase 5 Conclusion

The raw datasets are structurally suitable for further analysis, but they should
not yet be treated as analysis-ready.

Phase 5 identified specific data-quality issues and exceptions that must be
addressed or documented during Phase 6 data cleaning and transformation.

No assumptions will be made when handling ambiguous values. Issues will be
corrected only when supported by business rules, source information, or a
defined transformation rule.

In [99]:
# ============================================================
# STEP 14: FINAL DATA QUALITY SUMMARY
# ============================================================

phase5_summary = pd.DataFrame({
    "quality_dimension": [
        "Dataset Structure",
        "Completeness",
        "Key Uniqueness",
        "Referential Integrity",
        "Duplicate Records",
        "Categorical Consistency",
        "Distribution & Outliers",
        "Business Rule Validation"
    ],

    "status": [
        "PASS",
        "REVIEW",
        "PASS",
        "PASS",
        "REVIEW",
        "REVIEW",
        "REVIEW",
        "REVIEW"
    ],

    "phase5_assessment": [
        "11 related datasets identified",
        "Missing values exist in selected fields",
        "Candidate/master keys are unique",
        "Tested relationships show 100% matching records",
        "Geolocation contains duplicate records",
        "A small number of category inconsistencies identified",
        "Several variables contain strong skewness/outliers",
        "Several business-rule exceptions require review"
    ]
})

display(phase5_summary)

,quality_dimension,status,phase5_assessment
0,Dataset Structure,PASS,11 related datasets identified
1,Completeness,REVIEW,Missing values exist in selected fields
2,Key Uniqueness,PASS,Candidate/master keys are unique
3,Referential Integrity,PASS,Tested relationships show 100% matching records
4,Duplicate Records,REVIEW,Geolocation contains duplicate records
5,Categorical Consistency,REVIEW,A small number of category inconsistencies ide...
6,Distribution & Outliers,REVIEW,Several variables contain strong skewness/outl...
7,Business Rule Validation,REVIEW,Several business-rule exceptions require review


In [97]:
print("=" * 70)
print("PHASE 5 — DATA PROFILING FINAL STATUS")
print("=" * 70)

print("✓ Dataset structure profiling completed")
print("✓ Data-type profiling completed")
print("✓ Missing-value analysis completed")
print("✓ Key uniqueness analysis completed")
print("✓ Referential integrity analysis completed")
print("✓ Duplicate analysis completed")
print("✓ Distribution and outlier analysis completed")
print("✓ Categorical consistency analysis completed")
print("✓ Business-rule validation completed")
print("✓ Final data-quality assessment completed")

print("\nOverall Phase 5 Status: COMPLETED")
print("Next Phase: Data Cleaning & Transformation")
print("=" * 70)

PHASE 5 — DATA PROFILING FINAL STATUS
✓ Dataset structure profiling completed
✓ Data-type profiling completed
✓ Missing-value analysis completed
✓ Key uniqueness analysis completed
✓ Referential integrity analysis completed
✓ Duplicate analysis completed
✓ Distribution and outlier analysis completed
✓ Categorical consistency analysis completed
✓ Business-rule validation completed
✓ Final data-quality assessment completed

Overall Phase 5 Status: COMPLETED
Next Phase: Data Cleaning & Transformation
